# Vaardigheden week 9: kansrekening voor mensen die zich zullen vergissen

Dit notebook hoort bij het vaardighedenspoor. Het gebruikt dobbelstenen, een verzonnen bestelbus, een verzonnen fabriek en een verzonnen webwinkel, zodat de vaardigheden los blijven van het project van deze week.

Je meet een kans als een aandeel op de lange duur, gebruikt de drie regels waaraan elke kans gehoorzaamt, betrapt een AI-assistent die kansen optelt die niet opgeteld mogen worden, en berekent een voorwaardelijke kans in beide richtingen.

**Zo werk je ermee:** voer elke cel op volgorde uit met de afspeelknop of Shift+Enter. Lees de uitvoer voordat je verdergaat. Er zijn vier opdrachten onder "jouw beurt", en na elke opdracht staat een CHECK-cel.

## 1. Een kans is een aandeel op de lange duur

Gooi een eerlijke dobbelsteen heel vaak en tel de zessen. Het aandeel zessen springt eerst heen en weer en komt dan tot rust. Een kans is het aandeel dat een gebeurtenis op de lange duur inneemt, dus het is altijd een getal van 0 tot 1.

Het getal in `default_rng(9)` is een seed. Daardoor zijn de willekeurige getallen elke keer hetzelfde, zodat jouw resultaten gelijk zijn aan die van iedereen.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(9)
rolls = rng.integers(1, 7, size=100_000)       # 100.000 worpen met een eerlijke dobbelsteen
for n in [10, 100, 1_000, 10_000, 100_000]:
    share = (rolls[:n] == 6).mean()
    print(f"aandeel zessen in de eerste {n} worpen: {share:.4f}")
print("één op zes:", round(1 / 6, 4))

## 2. De complementregel en de somregel

Twee dobbelstenen hebben 36 uitkomsten, en elke uitkomst is even waarschijnlijk. De cel hieronder zet alle 36 in een tabel. De kans op een gebeurtenis is dan het aandeel rijen waarin die gebeurtenis voorkomt.

- **De complementregel zegt dat de kans op "niet A" gelijk is aan 1 min de kans op A.**
- **De somregel zegt dat voor twee gebeurtenissen die niet samen kunnen voorkomen, de kans op de ene of de andere de som van hun kansen is.** Een totaal van 7 en een dubbel kunnen niet bij dezelfde worp voorkomen, dus hun kansen tellen op.

In [ ]:
import itertools

dice = pd.DataFrame(list(itertools.product(range(1, 7), repeat=2)), columns=["first", "second"])
dice["total"] = dice["first"] + dice["second"]
print("rijen:", len(dice))

seven = dice["total"] == 7
double = dice["first"] == dice["second"]
print("P(totaal is 7):", round(seven.mean(), 4))
print("P(totaal is niet 7):", round((~seven).mean(), 4))
print("1 - P(totaal is 7):", round(1 - seven.mean(), 4))
print("P(dubbel):", round(double.mean(), 4))
print("P(7 of dubbel), geteld:", round((seven | double).mean(), 4))
print("P(7) + P(dubbel):", round(seven.mean() + double.mean(), 4))

Als twee gebeurtenissen wel samen kunnen voorkomen, telt optellen de overlap twee keer. "De eerste dobbelsteen toont 6" en "de tweede dobbelsteen toont 6" gebeuren allebei bij dubbel zes. De volledige somregel trekt de overlap af: P(A of B) = P(A) + P(B) - P(A en B).

In [ ]:
six_first = dice["first"] == 6
six_second = dice["second"] == 6
print("P(eerste is 6) + P(tweede is 6):", round(six_first.mean() + six_second.mean(), 4))
print("P(minstens één 6), geteld:", round((six_first | six_second).mean(), 4))
print("P(beide zijn 6), de overlap:", round((six_first & six_second).mean(), 4))
print("som min overlap:", round(six_first.mean() + six_second.mean() - (six_first & six_second).mean(), 4))

## 3. De productregel en onafhankelijkheid

Twee gebeurtenissen zijn onafhankelijk als het weten dat de ene gebeurde de kans op de andere niet verandert. Voor onafhankelijke gebeurtenissen is de kans dat beide gebeuren het product van hun kansen.

De twee dobbelstenen zijn onafhankelijk. De eerste dobbelsteen en het totaal zijn dat niet, want een 6 op de eerste dobbelsteen maakt een hoog totaal waarschijnlijker. De cel controleert beide paren.

In [ ]:
high = dice["total"] >= 10
print("P(eerste is 6) x P(tweede is 6):", round(six_first.mean() * six_second.mean(), 4))
print("P(beide zijn 6), geteld:", round((six_first & six_second).mean(), 4))
print()
print("P(eerste is 6) x P(totaal >= 10):", round(six_first.mean() * high.mean(), 4))
print("P(eerste is 6 en totaal >= 10), geteld:", round((six_first & high).mean(), 4))
print("P(totaal >= 10), alle worpen:", round(high.mean(), 4))
print("P(totaal >= 10 gegeven eerste is 6):", round(high[six_first].mean(), 4))

De laatste regel is een voorwaardelijke kans. "Gegeven dat de eerste dobbelsteen 6 is" betekent dat je alleen de rijen houdt waarin de eerste dobbelsteen 6 is, en daarbinnen het aandeel neemt. Paragraaf 8 komt op dit idee terug.

## 4. Overtuigingen die de rekenregels breken

Een uitgesproken kans is een overtuiging die als getal is opgeschreven, en de regels gelden ook voor overtuigingen. Een set overtuigingen die een regel breekt, heet incoherent. Van iemand met incoherente overtuigingen kun je laten zien dat hij geld verliest op weddenschappen die hij zelf eerlijk noemt.

Drie planners geven elk vijf kansen over twee gebeurtenissen. A is "de zending komt te laat" en B is "de weg naar de haven is afgesloten". De functie hieronder test vier regels en noemt de regels die een set breekt.

In [ ]:
def check_beliefs(b):
    """b bevat P(A), P(niet A), P(B), P(A en B) en P(A of B). Geeft de regels terug die de set breekt."""
    broken = []
    if any(not 0 <= v <= 1 for v in b.values()):
        broken.append("elke kans ligt tussen 0 en 1")
    if abs(b["A"] + b["not A"] - 1) > 1e-9:
        broken.append("P(A) + P(niet A) = 1")
    if b["A and B"] > min(b["A"], b["B"]):
        broken.append("P(A en B) is niet groter dan P(A) of P(B)")
    if abs(b["A or B"] - (b["A"] + b["B"] - b["A and B"])) > 1e-9:
        broken.append("P(A of B) = P(A) + P(B) - P(A en B)")
    return broken

planners = {
    "planner 1": {"A": 0.30, "not A": 0.70, "B": 0.20, "A and B": 0.10, "A or B": 0.40},
    "planner 2": {"A": 0.30, "not A": 0.60, "B": 0.20, "A and B": 0.10, "A or B": 0.40},
    "planner 3": {"A": 0.30, "not A": 0.70, "B": 0.20, "A and B": 0.25, "A or B": 0.25},
}
for name, b in planners.items():
    print(name, b)
    print("    breekt:", check_beliefs(b) or "niets, de set is coherent")

Planner 3 laat zien waarom je de regels samen moet controleren. De "A of B" van die planner past precies bij de somregel, en toch is de set incoherent, want twee gebeurtenissen kunnen niet vaker samen voorkomen dan een van beide überhaupt voorkomt.

## 5. Jouw beurt: herstel een vierde planner

Een vierde planner geeft de kansen hieronder. Voer de cel uit om te zien welke regel de set breekt. Sla daarna in de cel onder "jouw beurt" in `fixed_or` de waarde van P(A of B) op die de set coherent maakt, terwijl de andere vier kansen blijven zoals ze zijn.

In [ ]:
planner_4 = {"A": 0.50, "not A": 0.50, "B": 0.40, "A and B": 0.30, "A or B": 0.70}
print("planner 4 breekt:", check_beliefs(planner_4))

In [ ]:
# jouw beurt 1: sla in fixed_or de P(A of B) op die planner 4 coherent maakt
fixed_or = None

In [ ]:
# CHECK 1
assert fixed_or is not None, "fixed_or is nog leeg"
assert check_beliefs({**planner_4, "A or B": fixed_or}) == []
print("CHECK 1 geslaagd: planner 4 is coherent met P(A of B) =", round(fixed_or, 2))

## 6. Minstens één: de AI telt op wat niet opgeteld mag worden

Een bestelbus valt op 20 procent van de dagen uit, en de ene dag zegt niets over de volgende. We vroegen een AI-assistent: *"Wat is de kans dat de bus in een week van zeven dagen minstens één keer uitvalt?"* De assistent schreef de cel hieronder. Voer hem uit en beslis dan of het antwoord klopt.

In [ ]:
# Geschreven door een AI-assistent
p_day = 0.2
p_week = 7 * p_day
print("Kans op minstens één uitval in een week:", round(p_week, 2))

Een kans boven 1 is onmogelijk, dus het antwoord is fout. De assistent telde de zeven dagkansen op, en de somregel staat dat alleen toe voor gebeurtenissen die niet samen kunnen voorkomen. Uitval op maandag en uitval op dinsdag kunnen allebei gebeuren.

De veilige weg loopt via het complement. De bus komt de week alleen zonder uitval door als hij elk van de zeven dagen zonder uitval doorkomt. Voor onafhankelijke dagen vermenigvuldigt de productregel die zeven kansen. De kans op minstens één uitval is 1 min dat product. Een simulatie van 100.000 weken controleert de formule.

In [ ]:
p_no_breakdown_week = (1 - p_day) ** 7
p_at_least_one = 1 - p_no_breakdown_week
print("P(geen uitval de hele week):", round(p_no_breakdown_week, 4))
print("P(minstens één uitval):", round(p_at_least_one, 4))

rng = np.random.default_rng(19)
weeks = rng.random((100_000, 7)) < p_day        # één rij per week, één kolom per dag
print("gesimuleerd aandeel weken met uitval:", round(weeks.any(axis=1).mean(), 4))
print("het antwoord van de AI was te hoog met:", round(p_week - p_at_least_one, 2))
print("in natuurlijke frequenties: in ongeveer", round(100 * p_at_least_one), "van de 100 weken valt de bus uit")

Dezelfde fout over minder dagen geeft een antwoord dat mogelijk lijkt. Juist dat geval is gevaarlijk, want er staat geen getal boven 1 dat je waarschuwt. De cel vergelijkt de opgetelde kans met de juiste kans voor één tot zeven dagen, en de grafiek trekt de vergelijking door tot twee weken.

In [ ]:
import matplotlib.pyplot as plt

for n in [1, 2, 3, 5, 7]:
    print(f"{n} dagen: opgeteld {n * p_day:.2f}, juist {1 - (1 - p_day) ** n:.4f}")

days = np.arange(1, 15)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(days, days * p_day, marker="o", label="opgeteld (fout)")
ax.plot(days, 1 - (1 - p_day) ** days, marker="o", label="1 - (1 - p) ** n")
ax.axhline(1, color="grey", linestyle="--")
ax.set_title("Opgetelde dagkansen gaan boven 1, de juiste kans nooit")
ax.set_xlabel("aantal dagen")
ax.set_ylabel("kans op minstens één uitval")
ax.legend()
plt.tight_layout()
plt.show()

## 7. De kleine lettertjes van onafhankelijkheid

De productregel vraagt om onafhankelijke dagen. Veel echte gebeurtenissen komen juist in reeksen. Een bus die gisteren uitviel, kan vandaag nog steeds zwak zijn.

De volgende cel simuleert een tweede bus die in reeksen uitvalt. Na een dag met uitval is de kans op nieuwe uitval 50 procent. Na een goede dag is die kans 12,5 procent. Met deze twee getallen blijft het aandeel uitvaldagen op de lange duur 20 procent, net als bij de eerste bus. Voorspel het voordat je de cel uitvoert: heeft deze bus meer weken met uitval, of minder?

In [ ]:
rng = np.random.default_rng(29)
n_days = 7 * 100_000                            # 100.000 weken
u = rng.random(n_days)
broke = np.zeros(n_days, dtype=bool)
broke[0] = u[0] < 0.2
for t in range(1, n_days):
    broke[t] = u[t] < (0.5 if broke[t - 1] else 0.125)

spell_weeks = broke.reshape(-1, 7)              # één rij per week
print("aandeel dagen met uitval:", round(broke.mean(), 4))
print("aandeel weken met minstens één uitval:", round(spell_weeks.any(axis=1).mean(), 4))
print("onafhankelijkheidsformule voor hetzelfde dagaandeel:", round(1 - (1 - broke.mean()) ** 7, 4))

Beide bussen vallen op ongeveer één dag op vijf uit. De tweede bus heeft minder weken met uitval, omdat de uitval zich in dezelfde weken ophoopt en andere weken vrij laat. Als gebeurtenissen samenklonteren, overschat de onafhankelijkheidsformule de kans op minstens één. De oplossing is tellen: verdeel de gegevens in weken en neem het aandeel weken waarin de gebeurtenis voorkwam.

## 8. Jouw beurt: een functie voor "minstens één"

Schrijf een functie `at_least_one(p, n)` die de kans op minstens één gebeurtenis teruggeeft in `n` onafhankelijke pogingen, als elke poging kans `p` heeft. Je mag je AI-assistent erom vragen. Lees de code regel voor regel voordat je de CHECK uitvoert.

In [ ]:
# jouw beurt 2: schrijf de functie at_least_one(p, n)
def at_least_one(p, n):
    return None

In [ ]:
# CHECK 2
assert at_least_one(0.2, 7) is not None, "at_least_one geeft nog niets terug"
assert abs(at_least_one(0.2, 7) - (1 - 0.8 ** 7)) < 1e-9
assert abs(at_least_one(0.5, 1) - 0.5) < 1e-9
assert all(0 <= at_least_one(p, n) <= 1 for p in [0.1, 0.5, 0.9] for n in [1, 10, 100])
print("CHECK 2 geslaagd: at_least_one(0.2, 7) =", round(at_least_one(0.2, 7), 4))

## 9. Twee vragen die op elkaar lijken

Een verzonnen fabriek maakt 5.000 producten in een dagploeg en een nachtploeg, en sommige producten zijn defect. Een kruistabel telt elke combinatie van ploeg en status, met de totalen in de marges. Elke voorwaardelijke kans is een aandeel van één rij of één kolom van deze tabel.

In [ ]:
rng = np.random.default_rng(39)
n = 5_000
shift = np.where(rng.random(n) < 0.3, "night", "day")
broken_item = rng.random(n) < np.where(shift == "night", 0.08, 0.02)
items = pd.DataFrame({"shift": shift, "status": np.where(broken_item, "defective", "fine")})
table = pd.crosstab(items["shift"], items["status"], margins=True)   # night = nacht, day = dag, defective = defect, fine = goed
table

Twee vragen gebruiken hetzelfde aantal en delen het door verschillende totalen.

- *Welk aandeel van de producten uit de nachtploeg is defect?* Dat is P(defect gegeven nacht). Deel de defecten uit de nachtploeg door het totaal van de rij voor de nacht.
- *Welk aandeel van de defecte producten kwam uit de nachtploeg?* Dat is P(nacht gegeven defect). Deel hetzelfde aantal door het totaal van de kolom voor defect.

In [ ]:
night_defects = table.loc["night", "defective"]
p_def_given_night = night_defects / table.loc["night", "All"]
p_night_given_def = night_defects / table.loc["All", "defective"]
print("defecte producten uit de nachtploeg:", night_defects)
print("P(defect gegeven nacht):", round(p_def_given_night, 3))
print("P(nacht gegeven defect):", round(p_night_given_def, 3))
print("P(defect gegeven dag):", round(table.loc["day", "defective"] / table.loc["day", "All"], 3))

## 10. De assistent beantwoordt de andere vraag

Een manager vroeg een AI-assistent: *"Als een product uit de nachtploeg komt, hoe waarschijnlijk is het dan dat het defect is?"* Voer de code uit en vergelijk het antwoord met de tabel.

In [ ]:
# Geschreven door een AI-assistent
answer = (items.loc[items["status"] == "defective", "shift"] == "night").mean()
print("Kans dat een product uit de nachtploeg defect is:", round(answer, 3))

De assistent hield de defecte producten over en nam daarvan het aandeel uit de nachtploeg. Dat is P(nacht gegeven defect), het antwoord op de andere vraag. De verwisseling heeft een naam: de omgekeerde voorwaardelijke kans. In de rechtszaal heet ze de denkfout van de aanklager.

In [ ]:
print("de vraag van de manager, P(defect gegeven nacht):", round(p_def_given_night, 3))
print("de vraag die de AI beantwoordde, P(nacht gegeven defect):", round(answer, 3))
assert abs(answer - p_night_given_def) < 1e-12
print("het getal van de AI is P(nacht gegeven defect)")

## 11. Natuurlijke frequenties

Aandelen van aandelen zijn moeilijk uit elkaar te houden. Een veiliger vorm telt producten of mensen uit een rond getal, zoals 1.000. Die vorm heet natuurlijke frequenties, en daarin zien de twee vragen er net zo verschillend uit als ze zijn.

In [ ]:
print("Van 1.000 producten uit de nachtploeg zijn er ongeveer", round(1000 * p_def_given_night), "defect.")
print("Van 1.000 defecte producten komen er ongeveer", round(1000 * p_night_given_def), "uit de nachtploeg.")

Dezelfde vorm legt de denkfout van de aanklager bloot. Een stad heeft 500.000 volwassenen. De beschrijving van een getuige past bij één onschuldige volwassene op 1.000, en ook bij de persoon die het misdrijf pleegde. Een verdachte past bij de beschrijving. Een aanklager zegt: *"Maar één onschuldige op 1.000 zou passen, dus de kans dat deze verdachte onschuldig is, is één op 1.000."* Tel de mensen die passen.

In [ ]:
adults = 500_000
p_match_if_innocent = 1 / 1_000
innocent_matches = (adults - 1) * p_match_if_innocent
print(f"volwassenen in de stad: {adults:,}".replace(",", "."))
print("onschuldige volwassenen die bij de beschrijving passen:", round(innocent_matches))
print("schuldige volwassenen die passen: 1")
p_innocent_given_match = innocent_matches / (innocent_matches + 1)
print("P(past gegeven onschuldig):", p_match_if_innocent)
print("P(onschuldig gegeven past):", round(p_innocent_given_match, 4))

De aanklager noemde P(past gegeven onschuldig) en presenteerde die als P(onschuldig gegeven past). Alleen op grond van de overeenkomst blijven er in de stad ongeveer 500 onschuldige volwassenen over die even goed passen als de verdachte. De overeenkomst is nog steeds bewijs, maar er is ander bewijs naast nodig voordat ze naar één persoon wijst.

## 12. Jouw beurt: retouren van een webwinkel

Een verzonnen webwinkel noteert hoe elke bestelling is betaald, bij levering of met een kaart, en of de klant haar heeft teruggestuurd. Bereken beide voorwaardelijke kansen uit de tabel:

- `p_return_given_cod`, het aandeel van de bestellingen die bij levering zijn betaald en die zijn teruggestuurd;
- `p_cod_given_return`, het aandeel van de teruggestuurde bestellingen dat bij levering is betaald.

In [ ]:
rng = np.random.default_rng(49)
m = 3_000
payment = np.where(rng.random(m) < 0.4, "on delivery", "card")
sent_back = rng.random(m) < np.where(payment == "on delivery", 0.15, 0.05)
orders = pd.DataFrame({"payment": payment, "returned": np.where(sent_back, "returned", "kept")})
shop_table = pd.crosstab(orders["payment"], orders["returned"], margins=True)   # on delivery = bij levering, returned = teruggestuurd, kept = gehouden
shop_table

In [ ]:
# jouw beurt 3: bereken beide voorwaardelijke kansen uit de tabel orders
p_return_given_cod = None
p_cod_given_return = None

In [ ]:
# CHECK 3
assert p_return_given_cod is not None and p_cod_given_return is not None, "een waarde is nog leeg"
cod = orders["payment"] == "on delivery"
ret = orders["returned"] == "returned"
assert abs(p_return_given_cod - ret[cod].mean()) < 1e-9, "p_return_given_cod: houd eerst de bestellingen over die bij levering zijn betaald"
assert abs(p_cod_given_return - cod[ret].mean()) < 1e-9, "p_cod_given_return: houd eerst de teruggestuurde bestellingen over"
print("CHECK 3 geslaagd:", round(p_return_given_cod, 3), "en", round(p_cod_given_return, 3))

## 13. Jouw beurt: zet de stukken samen

Een chauffeur brengt 10 bestellingen die bij levering worden betaald naar één straat. Neem aan dat de 10 bestellingen onafhankelijk zijn. Gebruik je functie uit paragraaf 8 en je antwoord uit paragraaf 12 om in `p_any_return` de kans op te slaan dat minstens één van de 10 bestellingen terugkomt.

Schrijf daarna één zin in natuurlijke frequenties die de chauffeur zou begrijpen, beginnend met "Van 100 van zulke straten".

In [ ]:
# jouw beurt 4: de kans dat minstens één van 10 bestellingen die bij levering worden betaald, terugkomt
p_any_return = None

In [ ]:
# CHECK 4
assert p_any_return is not None, "p_any_return is nog leeg"
assert abs(p_any_return - (1 - (1 - ret[cod].mean()) ** 10)) < 1e-9
assert p_any_return < 10 * ret[cod].mean()
print("CHECK 4 geslaagd:", round(p_any_return, 3), "tegenover", round(10 * ret[cod].mean(), 3), "als de kansen waren opgeteld")